# RNNs and LSTMs

## Learning Objectives
1. Implement a vanilla RNN forward pass from scratch with numpy
2. Build a bidirectional LSTM for sequence classification in PyTorch
3. Train an LSTM for sentiment analysis on synthetic text data
4. Compare LSTM vs Transformer on sequential tasks across sequence lengths


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Vanilla RNN Forward Pass from Scratch

In [ ]:
# ── Level 1: manual RNN (numpy) ──────────────────────────────────────────
import numpy as np

np.random.seed(42)
INPUT_DIM  = 4    # feature dimension at each time step
HIDDEN_DIM = 8    # size of hidden state
SEQ_LEN    = 6    # sequence length

# Weight matrices
Wx = np.random.randn(HIDDEN_DIM, INPUT_DIM)  * 0.1   # input → hidden
Wh = np.random.randn(HIDDEN_DIM, HIDDEN_DIM) * 0.1   # hidden → hidden
b  = np.zeros(HIDDEN_DIM)

# Sample input sequence: (seq_len, input_dim)
x_seq = np.random.randn(SEQ_LEN, INPUT_DIM)

def rnn_forward(x_seq, Wx, Wh, b, h0=None):
    """Vanilla RNN: h_t = tanh(Wx @ x_t + Wh @ h_{t-1} + b)."""
    h = np.zeros(HIDDEN_DIM) if h0 is None else h0
    hidden_states = []
    for t, x_t in enumerate(x_seq):
        h = np.tanh(Wx @ x_t + Wh @ h + b)
        hidden_states.append(h.copy())
        print(f"  t={t}: h_t norm={np.linalg.norm(h):.4f}")
    return np.array(hidden_states), h   # all states, final state

print("RNN forward pass:")
all_h, h_final = rnn_forward(x_seq, Wx, Wh, b)
print(f"\nOutput shape (all hidden): {all_h.shape}   "
      f"Final hidden: {all_h[-1].shape}")
print(f"Hidden state range: [{all_h.min():.3f}, {all_h.max():.3f}]  "
      "(bounded by tanh ∈ [-1, 1])")


## Level 2 — Bidirectional LSTM in PyTorch

In [ ]:
# ── Level 2: LSTM sequence classification (torch) ────────────────────────
import torch, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)

# Synthetic: classify whether sum of even-indexed positions > 0
SEQ_LEN, VOCAB, EMBED, HIDDEN = 30, 200, 32, 64

def make_data(n=800, seq_len=SEQ_LEN):
    ids   = torch.randint(1, VOCAB, (n, seq_len))
    feats = ids.float() / VOCAB - 0.5          # normalised token values
    label = (feats[:, ::2].sum(dim=1) > 0).long()
    return ids, label

X_all, y_all = make_data(1000)
tr_ld = DataLoader(TensorDataset(X_all[:800], y_all[:800]), batch_size=64, shuffle=True)
X_v, y_v = X_all[800:].to(device), y_all[800:].to(device)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab, embed, hidden, n_class=2):
        super().__init__()
        self.embed  = nn.Embedding(vocab, embed, padding_idx=0)
        self.lstm   = nn.LSTM(embed, hidden, num_layers=2,
                               bidirectional=True, batch_first=True,
                               dropout=0.3)
        self.fc     = nn.Linear(hidden * 2, n_class)   # *2 for bidirectional

    def forward(self, x):
        e = self.embed(x)
        _, (h_n, _) = self.lstm(e)
        # Concatenate last forward + backward hidden states
        h = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        return self.fc(h)

model = BiLSTMClassifier(VOCAB, EMBED, HIDDEN).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Training BiLSTM for 20 epochs...")
for epoch in range(20):
    model.train()
    for xb, yb in tr_ld:
        xb, yb = xb.to(device), yb.to(device)
        try:
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("OOM: reduce batch_size or sequence length"); break
            raise

model.eval()
with torch.no_grad():
    acc = (model(X_v).argmax(1) == y_v).float().mean().item()
print(f"\nBiLSTM val accuracy: {acc:.3f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## Real-World Example 1 — LSTM Sentiment Analysis

In [ ]:
# ── RW1: LSTM for sentiment analysis ─────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Synthetic sentiment: positive reviews have higher token IDs on average
VOCAB, EMBED, HIDDEN, SEQ = 500, 64, 128, 40

def gen_reviews(n=1000):
    labels = torch.randint(0, 2, (n,))
    seqs   = []
    for lab in labels:
        if lab == 1:   # positive: tokens from upper half of vocab
            tokens = torch.randint(VOCAB // 2, VOCAB, (SEQ,))
        else:          # negative: tokens from lower half
            tokens = torch.randint(1, VOCAB // 2, (SEQ,))
        seqs.append(tokens)
    return torch.stack(seqs), labels

X_all, y_all = gen_reviews(1200)
loader = DataLoader(TensorDataset(X_all[:1000], y_all[:1000]),
                    batch_size=64, shuffle=True)
X_v, y_v = X_all[1000:].to(device), y_all[1000:].to(device)

class SentimentLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed  = nn.Embedding(VOCAB, EMBED, padding_idx=0)
        self.lstm   = nn.LSTM(EMBED, HIDDEN, batch_first=True)
        self.drop   = nn.Dropout(0.4)
        self.fc     = nn.Linear(HIDDEN, 2)

    def forward(self, x):
        _, (h, _) = self.lstm(self.embed(x))
        return self.fc(self.drop(h.squeeze(0)))

model = SentimentLSTM().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(15):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(X_v).argmax(1) == y_v).float().mean().item()
        print(f"Epoch {epoch+1:2d}  loss={total_loss/len(loader):.4f}  val_acc={acc:.3f}")


## Real-World Example 2 — Time-Series Forecasting with Stacked LSTM

In [ ]:
# ── RW2: stacked LSTM for next-N-step forecasting ────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# Synthetic sine-wave with noise
t      = np.linspace(0, 8 * np.pi, 2000)
signal = (np.sin(t) + 0.1 * np.random.randn(len(t))).astype(np.float32)

# Sliding window: predict next PRED_STEPS from LOOK_BACK context
LOOK_BACK, PRED_STEPS = 50, 10

def make_windows(s, lb, ps):
    xs, ys = [], []
    for i in range(len(s) - lb - ps):
        xs.append(s[i:i+lb])
        ys.append(s[i+lb:i+lb+ps])
    return (torch.tensor(xs).unsqueeze(-1),   # (N, lb, 1)
            torch.tensor(ys))                 # (N, ps)

X_all, y_all = make_windows(signal, LOOK_BACK, PRED_STEPS)
split = int(len(X_all) * 0.8)
loader = DataLoader(TensorDataset(X_all[:split], y_all[:split]),
                    batch_size=64, shuffle=True)
X_v, y_v = X_all[split:].to(device), y_all[split:].to(device)

class StackedLSTM(nn.Module):
    def __init__(self, in_dim=1, hidden=64, n_layers=2, pred_steps=PRED_STEPS):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=n_layers,
                            batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden, pred_steps)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])   # last layer's final hidden state

model = StackedLSTM().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        F.mse_loss(model(xb), yb).backward()
        opt.step()

model.eval()
with torch.no_grad():
    mse = F.mse_loss(model(X_v), y_v).item()
print(f"Stacked LSTM validation MSE: {mse:.6f}")
print("(Baseline: naive persistence MSE ≈ variance of target steps)")


## Real-World Example 3 — LSTM vs Transformer on Sequence Length

In [ ]:
# ── RW3: LSTM vs Transformer — accuracy vs sequence length ──────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import time

torch.manual_seed(42)

VOCAB, EMBED, HIDDEN = 100, 32, 64

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB, EMBED)
        self.lstm = nn.LSTM(EMBED, HIDDEN, batch_first=True)
        self.fc   = nn.Linear(HIDDEN, 2)
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        return self.fc(h.squeeze(0))

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB, EMBED)
        encoder_layer = nn.TransformerEncoderLayer(d_model=EMBED, nhead=4,
                                                    dim_feedforward=128,
                                                    batch_first=True)
        self.enc  = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc   = nn.Linear(EMBED, 2)
    def forward(self, x):
        out = self.enc(self.emb(x))
        return self.fc(out.mean(dim=1))   # mean pooling

def run_seq_len_experiment(seq_len, epochs=15):
    n = 600
    # Task: last token > VOCAB/2 → positive
    ids   = torch.randint(1, VOCAB, (n, seq_len))
    label = (ids[:, -1] > VOCAB // 2).long()
    ld    = DataLoader(TensorDataset(ids[:500], label[:500]),
                       batch_size=32, shuffle=True)
    Xv, yv = ids[500:].to(device), label[500:].to(device)

    results = {}
    for name, Model in [("LSTM", LSTMModel), ("Transformer", TransformerModel)]:
        m   = Model().to(device)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        t0  = time.perf_counter()
        for _ in range(epochs):
            m.train()
            for xb, yb in ld:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                F.cross_entropy(m(xb), yb).backward()
                opt.step()
        train_t = time.perf_counter() - t0
        m.eval()
        with torch.no_grad():
            acc = (m(Xv).argmax(1) == yv).float().mean().item()
        results[name] = (acc, train_t)
    return results

print(f"{'Seq Len':>8}  {'LSTM Acc':>9}  {'LSTM Time(s)':>13}  "
      f"{'Transf Acc':>11}  {'Transf Time(s)':>15}")
print("-" * 65)
for sl in [16, 64, 128, 256]:
    r = run_seq_len_experiment(sl)
    print(f"{sl:>8}  {r['LSTM'][0]:>9.3f}  {r['LSTM'][1]:>13.2f}  "
          f"{r['Transformer'][0]:>11.3f}  {r['Transformer'][1]:>15.2f}")
print("\nTransformer typically outperforms LSTM on longer sequences (parallelism + attention).")
